## SQL Book Data Exploration

My goal with this project is to showase my ability parsing real worth data within SQL Queries. I will be analyzing a dataset from GITHUB related to books, who made them, how they are rated across different platforms, and various other catagories on how the data is organized. 

LINK TO DATASETS USED IN THIS PROJECT --> https://openbookrank.com/en/open-data

In [1]:
import sqlite3
import pandas as pd
import matplotlib as plt

### In the cell below I have loaded my dataset and I am parsing through to find the overall shape and structure of the data within.

In [56]:
books = pd.read_csv("books.csv") # Reading the main CSV dataset
print(f'{books.shape} Rows, Columns') # Returns the amount of rows and columns in the file.
print('')
print(f'Column Names: {books.columns}') # Returns a list of column names, which will be essential for SQL parsing   
print('')
print(books.isna().sum()) # Returns how many NA values are in each column of the dataset (WHERE NOT NULL)

(12984, 17) Rows, Columns

Column Names: Index(['id', 'slug', 'title', 'author', 'year', 'score', 'ratings',
       'confidence', 'popularity_percentile', 'best_rank',
       'best_rank_category', 'series', 'series_position', 'wikidata',
       'openlibrary', 'isbn13', 'url'],
      dtype='object')

id                          0
slug                        0
title                       0
author                    396
year                      941
score                    8636
ratings                     0
confidence               8636
popularity_percentile       0
best_rank                9017
best_rank_category       9017
series                   2130
series_position          5088
wikidata                  753
openlibrary              7281
isbn13                   6097
url                         0
dtype: int64


### Below is a small visualization of how the data itself looks. Important to note is the 'slug' column which will be used as an ID for future JOINs

In [49]:
print(books.head)

<bound method NDFrame.head of                                                       id  \
0                                       obr:work:1-d395b   
1                          obr:work:1-lifesteal-volume-2   
2                              obr:work:10th-anniversary   
3                                      obr:work:11-22-63   
4      obr:work:11th-and-12th-centuries-from-emperor-...   
...                                                  ...   
12979                                  obr:work:zone-one   
12980                   obr:work:zorba-the-hutts-revenge   
12981                         obr:work:zu-ihrem-gedenken   
12982                    obr:work:zuhaeltertum-in-berlin   
12983                                obr:work:zwarte-zee   

                                                    slug  \
0                                                1-d395b   
1                                   1-lifesteal-volume-2   
2                                       10th-anniversary   
3        

In [78]:
for name in ["books", "authors", "series", "rankings", "identifiers"]:
    data = pd.read_csv(f"{name}.csv")
    print(f"{name} Table's Columns: {data.columns}")
    print('')

books Table's Columns: Index(['id', 'slug', 'title', 'author', 'year', 'score', 'ratings',
       'confidence', 'popularity_percentile', 'best_rank',
       'best_rank_category', 'series', 'series_position', 'wikidata',
       'openlibrary', 'isbn13', 'url'],
      dtype='object')

authors Table's Columns: Index(['id', 'slug', 'name', 'wikidata', 'ranked_books', 'series', 'url'], dtype='object')

series Table's Columns: Index(['id', 'slug', 'title', 'author', 'volumes', 'score', 'ratings', 'url'], dtype='object')

rankings Table's Columns: Index(['category', 'rank', 'work', 'slug', 'title', 'author', 'score',
       'ratings'],
      dtype='object')

identifiers Table's Columns: Index(['scheme', 'value', 'work', 'slug'], dtype='object')



### The code below links my datasets to sqlite3, and verifies that I can make SQL requests.

In [31]:
conn = sqlite3.connect("bookrank.db")

for name in ["books", "authors", "series", "rankings", "identifiers"]:
    df = pd.read_csv(f"{name}.csv")
    df.to_sql(name, conn, if_exists="replace", index=False)

In [68]:
tables_query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql_query(tables_query, conn)

,name
0,books
1,authors
2,series
3,rankings
4,identifiers


## The code below creates a small snippet of each different dataset I am using. This is so I can reference these tables in future SQL queries and find the specific columns/rows of data I need.

In [128]:
for name in ["books", "authors", "series", "rankings", "identifiers"]:
    
    test_query = f"""
    SELECT * 
    FROM {name}
    LIMIT 5
    """
    
    print(name)
    display(pd.read_sql_query(test_query, conn))

books


,id,slug,title,author,year,score,ratings,confidence,popularity_percentile,best_rank,best_rank_category,series,series_position,wikidata,openlibrary,isbn13,url
0,obr:work:1-d395b,1-d395b,"Berserk, Volume 1",Kentaro Miura,1990.0,NaN,27,NaN,62,NaN,None,berserk,1,Q140148656,OL8988597W,9781630089993,https://openbookrank.com/en/book/1-d395b
1,obr:work:1-lifesteal-volume-2,1-lifesteal-volume-2,1% Lifesteal (Volume 2),Robert Blaise,2025.0,7.96,45,0.228,66,NaN,None,1-lifesteal,2,None,HC1838371,None,https://openbookrank.com/en/book/1-lifesteal-v...
2,obr:work:10th-anniversary,10th-anniversary,10th Anniversary,James Patterson,2011.0,NaN,40,NaN,65,NaN,None,womens-murder-club,10,Q19800952,OL15719783W,None,https://openbookrank.com/it/libro/10th-anniver...
3,obr:work:11-22-63,11-22-63,11/22/63,Stephen King,2011.0,8.62,1874,0.925,99,16.0,science-fiction,None,None,Q723020,OL16002468W,9781451627299,https://openbookrank.com/en/book/11-22-63
4,obr:work:11th-and-12th-centuries-from-emperor-...,11th-and-12th-centuries-from-emperor-henry-ii-...,11th and 12th Centuries: From Emperor Henry II...,Karlheinz Deschner,1999.0,NaN,0,NaN,0,NaN,None,kriminalgeschichte-des-christentums,6,Q137806847,None,None,https://openbookrank.com/en/book/11th-and-12th...


authors


,id,slug,name,wikidata,ranked_books,series,url
0,obr:author:a-a-milne,a-a-milne,A. A. Milne,Q207036,3,1,https://openbookrank.com/en/author/a-a-milne
1,obr:author:a-c-crispin,a-c-crispin,A. C. Crispin,None,1,1,https://openbookrank.com/en/author/a-c-crispin
2,obr:author:a-s-byatt,a-s-byatt,A. S. Byatt,Q232417,1,0,https://openbookrank.com/en/author/a-s-byatt
3,obr:author:a-w-tozer,a-w-tozer,A. W. Tozer,None,1,0,https://openbookrank.com/en/author/a-w-tozer
4,obr:author:abby-jimenez,abby-jimenez,Abby Jimenez,None,4,1,https://openbookrank.com/en/author/abby-jimenez


series


,id,slug,title,author,volumes,score,ratings,url
0,obr:series:1-lifesteal,1-lifesteal,1% Lifesteal,Robert Blaise,5,7.76,55,https://openbookrank.com/en/series/1-lifesteal
1,obr:series:1632-series,1632-series,1632 series,Eric Flint,23,7.81,90,https://openbookrank.com/en/series/1632-series
2,obr:series:1q84,1q84,1Q84,Haruki Murakami,3,7.52,896,https://openbookrank.com/en/series/1q84
3,obr:series:30-days-of-night,30-days-of-night,30 Days of Night,Jeff Mariotte,3,NaN,0,https://openbookrank.com/en/series/30-days-of-...
4,obr:series:7-secondes,7-secondes,7 secondes,Jean-David Morvan,3,NaN,0,https://openbookrank.com/en/series/7-secondes


rankings


,category,rank,work,slug,title,author,score,ratings
0,adventure,1,obr:work:harry-potter-and-the-prisoner-of-azkaban,harry-potter-and-the-prisoner-of-azkaban,Harry Potter and the Prisoner of Azkaban,J. K. Rowling,8.78,8313
1,adventure,2,obr:work:skyward,skyward,Skyward,Brandon Sanderson,8.73,1314
2,adventure,3,obr:work:the-crippled-god,the-crippled-god,The Crippled God,Steven Erikson,8.71,278
3,adventure,4,obr:work:the-last-olympian,the-last-olympian,The Last Olympian,Rick Riordan,8.67,2010
4,adventure,5,obr:work:the-house-of-hades,the-house-of-hades,The House of Hades,Rick Riordan,8.64,1043


identifiers


,scheme,value,work,slug
0,isbn13,9781630089993,obr:work:1-d395b,1-d395b
1,wikidata,Q140148656,obr:work:1-d395b,1-d395b
2,openlibrary,OL8988597W,obr:work:1-d395b,1-d395b
3,openlibrary,HC1838371,obr:work:1-lifesteal-volume-2,1-lifesteal-volume-2
4,hardcover,1-lifesteal-volume-2,obr:work:1-lifesteal-volume-2,1-lifesteal-volume-2


### Now that the sqlite3 is running with my data, its time to begin solving some problems

## Question 1:
### What Are the top 5 Rated Books of each Category?

In [183]:
# First step is to find all of the different categories
# There are 46 categories which is way too many, so I'll see If I can find the most 5 most popular categories
# My naming convention I will use for these queries is QUESTION[number]_QUERY[number], so this first one is Question 1 Query 1, or q1_q1
q1_q1 = """ 
SELECT DISTINCT
category,
count(*) AS num_books
FROM
rankings
GROUP BY category
ORDER BY num_books DESC
LIMIT 5
"""
pd.read_sql_query(q1_q1, conn)

,category,num_books
0,fantasy,1344
1,science-fiction,761
2,crime,448
3,young-adult,381
4,classics,355


In [197]:
for category in ["fantasy","science-fiction","crime","young-adult","classics"]:
    q1_q2 = f"""
    SELECT 
    b.title,
    r.score,
    b.confidence,
    r.ratings
    FROM books AS b
    JOIN rankings as r
    on b.slug = r.slug
    WHERE r.category = '{category}' AND b.confidence > 0.9
    ORDER BY r.score DESC
    LIMIT 5
    """
    print(category)
    display(pd.read_sql_query(q1_q2, conn))

fantasy


,title,score,confidence,ratings
0,Words of Radiance,9.36,0.958,3473
1,The Way of Kings,9.20,0.963,3927
2,Mistborn: The Hero of Ages,9.10,0.965,4191
3,Oathbringer,9.07,0.942,2461
4,The Lord of the Rings,9.04,0.933,2111


science-fiction


,title,score,confidence,ratings
0,Project Hail Mary,8.99,0.982,8300
1,Network Effect,8.92,0.912,1585
2,The Martian,8.84,0.976,6086
3,Golden Son,8.84,0.948,2753
4,Dune,8.82,0.978,6680


crime


,title,score,confidence,ratings
0,Crime and Punishment,8.46,0.927,1928
1,And Then There Were None,8.42,0.946,2672
2,A Good Girl's Guide to Murder,8.29,0.910,1537
3,The Secret History,8.16,0.925,1870
4,Where the Crawdads Sing,8.15,0.922,1787


young-adult


,title,score,confidence,ratings
0,Harry Potter and the Deathly Hallows,8.82,0.979,7068
1,Harry Potter and the Goblet of Fire,8.82,0.980,7473
2,Harry Potter and the Prisoner of Azkaban,8.78,0.982,8313
3,Harry Potter and the Half-Blood Prince,8.75,0.978,6737
4,Queen of Shadows,8.74,0.908,1492


classics


,title,score,confidence,ratings
0,The Lord of the Rings,9.04,0.933,2111
1,The Two Towers,8.89,0.962,3852
2,Dune,8.82,0.978,6680
3,The Fellowship of the Ring,8.80,0.975,5853
4,Flowers for Algernon,8.79,0.943,2493


### SQL BREAKDOWNS



The goal of my first SQL query was to find all of the categories of books and see which categories had the highest numbers of books within them. I figured these categories would be the most interesting to select for the next step of my question, which was finding the highest rated books across these categories. 

SELECT DISTINCT \
category, \
count(*) AS num_books

This SELECT statement uses 'DISTINCT', because I only need one entry for each category of books. 'category' Is just pulling the name of the book category, and 'count(*) AS num_books' is counting the number of books within this category. With this select statement the returned query should be 'name of category' | 'number of books in category'.

FROM \
rankings

This is just the sepecific dataset I want to pull data from, as the rankings table holds category information within it.

GROUP BY category \
ORDER BY num_books DESC \
LIMIT 5

The 'GROUP BY category' groups the 'count(*)' (number of books) into each category, this is essentially the most important part of the SQL, as it is what sorts my data into categories and book counts. The 'ORDER BY num_books DESC' Then takes this grouped data and sorts it starting with categories which have the highest 'num_books', remember, I'm looking for the 5 categories with the most books. 'LIMIT 5' simply just selects the first five rows of the returned query.

The goal of my second query was to then use these 5 categories, and find which books had the highest rating for each category. Important to note is the confidence column, which notes how confident the creators of the dataset are in the ratings of each book. This number is calculated with a formula which takes into account the amount of consistent ratings from differing sites. I also combined some python logic with this query, in order to make different tables of each category with their most popular books. 

SELECT \
b.title, \
r.score, \
b.confidence, \
r.ratings

This select stament returns a book's name, its score, confidence rating, and how many ratings it has across platforms. Notice the 'b.' and 'r.', which are aliases for the book table and rankings table. 

FROM books AS b \
JOIN rankings as r \
on b.slug = r.slug 

These FROM and JOIN statements tell my query where to pull data from. I started with the books table as it has a lot of key information (such as book names), then joined it with the rankings table which included the score and ranking data I wanted. I joined them on 'slug', which was a common column both tables shared, and slugs work as a unique identifier for each book. 

WHERE r.category = '{category}' AND b.confidence > 0.9 \
ORDER BY r.score DESC \
LIMIT 5

This part of my query is where the data is grouped, sorted, and filtered. The WHERE statement used some python logic, and makes it so that the query only selects data for the specific category of the for loop. I wanted the ratings to also be accurate, so I only selected book that have a high confidence rating (above 0.9). The data is then sorted by ranking, as I want to pull only the books with the highest rankings, and I put a limit of 5 to only select the 5 books with the highest rankings. 

## Question 2:
### Which Authors have the most consistent high ratings?

In [226]:
# This SQL will mainly require using a GROUP BY on the Books table, because it includes author names and the slug I need to connect with the rankings table.
# This is also a good question because it will require using a HAVING clause, which will sort out authors who have not published enough books.

q2_q1 = '''
SELECT
b.author,
ROUND(AVG(b.score), 2) AS avg_score,
COUNT(*) AS 'num_books_ranked',
SUM(b.ratings) AS total_ratings
FROM books AS b
WHERE b.author IS NOT NULL AND b.score IS NOT NULL
GROUP BY b.author
HAVING COUNT(*) > 3
ORDER BY avg_score DESC
LIMIT 10
'''
pd.read_sql_query(q2_q1, conn)

,author,avg_score,num_books_ranked,total_ratings
0,Matt Dinniman,8.69,9,16037
1,Pierce Brown,8.64,6,11924
2,Larry McMurtry,8.49,4,847
3,Bill Watterson,8.47,4,259
4,Carl Sagan,8.45,4,1280
5,Shel Silverstein,8.43,4,2668
6,Will Wight,8.42,15,5448
7,Steven Erikson,8.41,12,4861
8,Alan Moore,8.41,5,3672
9,Joe Abercrombie,8.39,12,9466


### SQL BREAKDOWN

SELECT \
b.author, \
ROUND(AVG(b.score), 2) AS avg_score, \
COUNT(*) AS 'num_books_ranked', \
SUM(b.ratings) AS total_ratings

The SELECT part of this query is probably the most complex and important part of the query, so here's what each section does individually. 'b.author' pulls the authors name. 'ROUND(AVG(b.score), 2)' Pulls the average score of the books which the author has written. This statement does not work on its own, but when paired with the GROUP BY statement will pull the average book score across each author's published books. 'COUNT(*)' Pulls the number of books which each different author has written. 'SUM(b.ratings)' Pulls the total amount of ratings each author has recived across all of their books.

FROM books AS b 

This SQL query only required me to use data from the books table. 

WHERE b.author IS NOT NULL AND b.score IS NOT NULL \
GROUP BY b.author \
HAVING COUNT(*) > 3 \
ORDER BY avg_score DESC

This part of the query is where I sort and filtered the results. I wanted to make sure I was not pulling any data from non-existent authors, so I filtered them out of the result. Same for the score, I wanted to make sure each author had valid scores attached to their books, as to not sway the data in any unusual ways. The GROUP BY statement groups all my data by author. This means that the average rating of their books, how many books they've published, and how many reviews their books reciveved will all be grouped together by each individual author. The HAVING addition at the end of the GROUP BY just ensures that authors have published more than 3 books, which I feel is a fair amount to begin averaging ratings for. The ORDER BY simply just organized by highest average book score to lowest, and is essential as I am looking for authors with the highest average book ratings. 